In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Impostazioni grafiche per il report
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12})

# =============================================================================
# 1. INSERIMENTO DATI (Inserisci qui i tuoi numeri finali dello Step 3)
# =============================================================================
# Ho messo i valori che abbiamo visto insieme, aggiornali se ne hai di più precisi.
results_data = [
    # --- LSTM MODELS ---
    {'Model': 'LSTM Best (Win=10)',   'Type': 'LSTM',        'MSE': 0.03617}, # Il tuo campione
    {'Model': 'LSTM Light',           'Type': 'LSTM',        'MSE': 0.04500}, # (Esempio: metti il tuo valore)
    {'Model': 'LSTM HighReg',         'Type': 'LSTM',        'MSE': 0.04100}, # (Esempio)
    {'Model': 'LSTM Long (Win=20)',   'Type': 'LSTM',        'MSE': 0.03900}, # (Esempio)
    
    # --- TRANSFORMER MODELS ---
    {'Model': 'Trans Standard',       'Type': 'Transformer', 'MSE': 0.08650}, # Il valore che scendeva piano
    {'Model': 'Trans Complex',        'Type': 'Transformer', 'MSE': 0.22000}, # (Se lo hai fatto)
    
    # --- BASELINE ---
    {'Model': 'Baseline (Inertia)',   'Type': 'Baseline',    'MSE': 0.14110}  # Il riferimento
]

# Creazione DataFrame
df_results = pd.DataFrame(results_data).sort_values(by='MSE', ascending=True)

# =============================================================================
# 2. TABELLA 1 (Model Selection)
# =============================================================================
print("\n" + "="*60)
print("TABLE 1: MODEL SELECTION & COMPARISON")
print("="*60)
print(df_results.to_string(index=False))
print("-" * 60)

# Salva la tabella in CSV per poterla copiare in Excel/Word
df_results.to_csv("Table_1_Model_Selection.csv", index=False)
print("✅ Tabella salvata in 'Table_1_Model_Selection.csv'")

# =============================================================================
# 3. GRAFICO (Bar Plot Comparativo)
# =============================================================================
plt.figure(figsize=(12, 6))

# Creiamo il barplot con colori diversi per Tipo
barplot = sns.barplot(
    data=df_results, 
    x='Model', 
    y='MSE', 
    hue='Type', 
    palette={'LSTM': '#2ecc71', 'Transformer': '#e74c3c', 'Baseline': '#95a5a6'},
    edgecolor='black'
)

# Aggiungiamo i numeri sopra le barre
for p in barplot.patches:
    if p.get_height() > 0: # Evita errori su barre vuote
        barplot.annotate(format(p.get_height(), '.4f'), 
                         (p.get_x() + p.get_width() / 2., p.get_height()), 
                         ha = 'center', va = 'center', 
                         xytext = (0, 9), 
                         textcoords = 'offset points',
                         fontsize=10, weight='bold')

plt.title('Model Selection: MSE Comparison on Test Set', fontsize=15, weight='bold')
plt.xlabel('Model Configuration', fontsize=12)
plt.ylabel('Mean Squared Error (Lower is Better)', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='Architecture')
plt.tight_layout()

# Salvataggio alta risoluzione
plt.savefig("Evaluation_0_Model_Comparison.jpg", dpi=300)
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

# === SETUP GRAFICO ===
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 12})

# === 1. CARICAMENTO MODELLO E DATI ===
# Percorsi (Assicurati che siano corretti rispetto alle tue cartelle)
DATA_PATH = "processed_step2_triple_head"
MODEL_PATH = "saved_models_dynamics_lstm/LSTM_Best_Run/best_model.keras" 

print(f"📥 Caricamento dati da: {DATA_PATH}")
test_df = pd.read_csv(os.path.join(DATA_PATH, "test_dataset.csv"))
test_data = test_df.values.astype(np.float32)

print(f"🧠 Caricamento modello LSTM: {MODEL_PATH}")
model = load_model(MODEL_PATH)

# === 2. PREPARAZIONE DATI TEST (Logica 'Target Assoluto') ===
def create_sequences_eval(data, seq_length=10):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        xs.append(data[i : i + seq_length])
        ys.append(data[i + seq_length]) # Target: Valore futuro assoluto
    return np.asarray(xs), np.asarray(ys)

X_test, y_true = create_sequences_eval(test_data, seq_length=10)

print("🔮 Generazione predizioni sul Test Set...")
y_pred = model.predict(X_test, verbose=1)

# =============================================================================
# ANALISI A: METRICHE PER DIMENSIONE (Discover Biases)
# =============================================================================
n_dims = y_true.shape[1]
r2_scores = []
mse_scores = []

# Calcoliamo R2 per ogni singola colonna (dimensione latente)
for i in range(n_dims):
    r2 = r2_score(y_true[:, i], y_pred[:, i])
    mse = mean_squared_error(y_true[:, i], y_pred[:, i])
    r2_scores.append(r2)
    mse_scores.append(mse)

# Plotting R2 Score
plt.figure(figsize=(14, 6))
x_axis = np.arange(n_dims)
# Usiamo un gradiente di colore basato sul punteggio
bars = plt.bar(x_axis, r2_scores, color=plt.cm.viridis(np.array(r2_scores)))

plt.axhline(y=0, color='r', linestyle='-', linewidth=1) # Linea dello zero
plt.title(f'Evaluation 1.A: Learning Performance by Latent Dimension (R² Score)', fontsize=16)
plt.xlabel('Latent Dimension Index')
plt.ylabel('R² Score (1.0 = Perfect, 0.0 = Baseline, <0 = Bad)')
plt.ylim(bottom=min(min(r2_scores), -0.1), top=1.05) # Zoomma per vedere bene

# Colorbar fittizia per estetica
sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=min(r2_scores), vmax=1))
plt.colorbar(sm, label='Performance Intensity')

plt.tight_layout()
plt.savefig("Evaluation_1A_R2_by_Dimension.jpg", dpi=300)
plt.show()

print(f"✅ Analisi Bias completata. Dimensioni con R2 < 0 (non imparate): {sum(np.array(r2_scores) < 0)}")


# =============================================================================
# ANALISI B: TIME SERIES DYNAMICS (Capire la Fisica)
# =============================================================================
# Prendiamo le 3 dimensioni con la varianza più alta (quelle che si muovono di più)
# Spesso le prime dimensioni dell'Autoencoder sono le più importanti
dims_to_plot = [0, 1, 2] 
zoom_start, zoom_end = 0, 300 # Guardiamo i primi 300 step

plt.figure(figsize=(15, 10))
for i, dim in enumerate(dims_to_plot):
    plt.subplot(3, 1, i+1)
    
    # Dati Reali vs Predetti
    plt.plot(y_true[zoom_start:zoom_end, dim], label='Ground Truth (Latent)', color='black', linewidth=2, alpha=0.6)
    plt.plot(y_pred[zoom_start:zoom_end, dim], label='LSTM Prediction', color='#00cc96', linewidth=2, linestyle='--')
    
    plt.title(f'Latent Dimension {dim} Dynamics (First {zoom_end} steps)')
    plt.ylabel('Value')
    if i == 0: plt.legend(loc='upper right')

plt.xlabel('Time Step')
plt.tight_layout()
plt.savefig("Evaluation_1B_TimeSeries.jpg", dpi=300)
plt.show()


# =============================================================================
# ANALISI C: SCATTER PLOT & BIAS CHECK (Correlazione)
# =============================================================================
plt.figure(figsize=(8, 8))

# Flattening: Mettiamo tutti i punti di tutte le dimensioni insieme per vedere la qualità globale
plt.scatter(y_true.flatten(), y_pred.flatten(), alpha=0.05, color='#3498db', s=1)

# Linea ideale (x=y)
min_val = min(y_true.min(), y_pred.min())
max_val = max(y_true.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Ideal Prediction')

plt.title('Evaluation 1.C: Global Correlation (Prediction vs Truth)')
plt.xlabel('True Latent Value')
plt.ylabel('Predicted Latent Value')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig("Evaluation_1C_Scatter_Bias.jpg", dpi=300)
plt.show()

In [ ]:
# =============================================================================
# EVALUATION 2: PHYSICAL SPACE VERIFICATION (End-to-End)
# =============================================================================
from sklearn.metrics import mean_squared_error

# 1. CARICAMENTO DECODER (Dallo Step 2)
# ⚠️ IMPORTANTE: Controlla se il file esiste in questa cartella!
DECODER_PATH = "saved_models/decoder_best.keras" 

print(f"🔓 Tentativo di caricamento Decoder da: {DECODER_PATH}")

try:
    decoder = load_model(DECODER_PATH)
    print("✅ Decoder caricato! Possiamo tornare allo spazio fisico.")
    
    # 2. DECODIFICA (Latent -> Physical)
    # Trasformiamo le predizioni della LSTM (numeri compressi) in atomi
    print("⚙️ Decodifica in corso (può richiedere qualche secondo)...")
    X_pred_phys = decoder.predict(y_pred, verbose=0)
    X_true_phys = decoder.predict(y_true, verbose=0) # Decodifichiamo anche il target per confronto equo
    
    # 3. CALCOLO ERRORE FISICO (RMSE)
    # RMSE = Root Mean Squared Error. È l'errore medio in "unità di spazio".
    # Se i dati erano scalati (es. StandardScaler), questo è l'errore in deviazioni standard.
    # Se erano in Angstrom, è l'errore in Angstrom.
    mse_phys = mean_squared_error(X_true_phys, X_pred_phys)
    rmse_phys = np.sqrt(mse_phys)
    
    print("\n" + "="*50)
    print(f"🌍 RISULTATO END-TO-END (Physical Space)")
    print("="*50)
    print(f"📉 RMSE Atomico Medio: {rmse_phys:.5f}")
    print("="*50)
    
    # 4. PLOT TRAIETTORIE FISICHE (Reale vs Ricostruito)
    # Prendiamo la coordinata X del PRIMO ATOMO (indice 0)
    # e la coordinata Y del PRIMO ATOMO (indice 1) per vedere se si muove bene.
    
    plt.figure(figsize=(15, 6))
    
    # Sottografico 1: Coordinata X Atomo 1
    plt.subplot(1, 2, 1)
    plt.plot(X_true_phys[:200, 0], label='Real Atom (X)', color='black', linewidth=2, alpha=0.7)
    plt.plot(X_pred_phys[:200, 0], label='Predicted Atom (X)', color='#e67e22', linestyle='--', linewidth=2)
    plt.title("Physical Trajectory: Atom 1 (X-Coord)")
    plt.ylabel("Position (Scaled)")
    plt.xlabel("Time Step")
    plt.legend()
    
    # Sottografico 2: Coordinata Y Atomo 1 (Per conferma)
    plt.subplot(1, 2, 2)
    plt.plot(X_true_phys[:200, 1], label='Real Atom (Y)', color='black', linewidth=2, alpha=0.7)
    plt.plot(X_pred_phys[:200, 1], label='Predicted Atom (Y)', color='#e67e22', linestyle='--', linewidth=2)
    plt.title("Physical Trajectory: Atom 1 (Y-Coord)")
    plt.xlabel("Time Step")
    plt.legend()
    
    plt.tight_layout()
    plt.savefig("Evaluation_2_Physical_Space.jpg", dpi=300)
    plt.show()

except Exception as e:
    print(f"\n❌ ERRORE CRITICO: Non riesco a caricare il Decoder o a decodificare.")
    print(f"Dettaglio errore: {e}")
    print("Assicurati di aver eseguito lo Step 2 e di avere il file 'decoder_best.keras'.")

In [ ]:
# =============================================================================
# EVALUATION 3: GENERATIVE STABILITY (The "Ghost Protein" Test)
# =============================================================================
from sklearn.decomposition import PCA

# 1. CONFIGURAZIONE DEL LOOP
# Quanti step vogliamo generare nel futuro? (Es. 1000 = generazione lunga)
GENERATION_STEPS = 1000 
SEQ_LEN = 10  # La tua window size

# 2. SELEZIONE PUNTO DI PARTENZA
# Prendiamo una finestra a caso dal Test Set per iniziare
start_idx = 0 
initial_window = X_test[start_idx] # Shape: (10, Latent_Dim)

print(f"🔮 Avvio generazione autoregressiva per {GENERATION_STEPS} step...")
print("   (Il modello si nutre delle sue stesse predizioni...)")

# 3. IL LOOP DI GENERAZIONE (Autoregressive Loop)
current_window = initial_window.copy()
generated_trajectory = []

for _ in range(GENERATION_STEPS):
    # a. Prepara l'input (Shape: 1, 10, Latent)
    input_reshaped = current_window.reshape(1, SEQ_LEN, -1)
    
    # b. Predici il prossimo step (t+1)
    next_step = model.predict(input_reshaped, verbose=0) # Shape: (1, Latent)
    
    # c. Salva la predizione
    generated_trajectory.append(next_step[0])
    
    # d. Aggiorna la finestra (Rolling Window)
    # Togli il primo frame (vecchio) e aggiungi quello appena predetto (nuovo)
    # Stack verticale: [Old_Window[1:]] + [New_Prediction]
    current_window = np.vstack([current_window[1:], next_step[0]])

# Convertiamo la lista in array numpy
gen_data = np.array(generated_trajectory)
print(f"✅ Generazione completata! Shape traiettoria: {gen_data.shape}")


# =============================================================================
# 4. PLOT: PHASE PORTRAIT (PC1 vs PC2)
# =============================================================================
# Usiamo la PCA per proiettare tutto in 2D e vedere la "Mappa" della proteina
print("📊 Calcolo PCA per il Phase Portrait...")

pca = PCA(n_components=2)
# Addestriamo la PCA sui dati VERI (per capire lo spazio legittimo)
pca.fit(y_true) 

# Trasformiamo i dati Veri (Neri) e Generati (Rossi)
y_true_pca = pca.transform(y_true)
gen_data_pca = pca.transform(gen_data)

plt.figure(figsize=(10, 10))

# A. Background: La Realtà (Nuvola Nera)
# Questo rappresenta tutti gli stati fisici possibili della proteina
plt.scatter(y_true_pca[:, 0], y_true_pca[:, 1], 
            c='black', alpha=0.05, s=10, label='Real Physics (Manifold)')

# B. Foreground: Il Sogno (Linea Rossa)
# Questa è la strada che la tua IA ha deciso di percorrere da sola
plt.plot(gen_data_pca[:, 0], gen_data_pca[:, 1], 
         c='#e74c3c', linewidth=2, label='Generated Trajectory (Autoregressive)')

# Segniamo l'inizio e la fine
plt.scatter(gen_data_pca[0, 0], gen_data_pca[0, 1], c='green', s=100, edgecolors='white', label='Start')
plt.scatter(gen_data_pca[-1, 0], gen_data_pca[-1, 1], c='blue', s=100, edgecolors='white', label='End')

plt.title(f"Evaluation 3: Generative Stability (Phase Portrait)\nPC1 vs PC2 - {GENERATION_STEPS} Steps", fontsize=14)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig("Evaluation_3_Stability_PhaseSpace.jpg", dpi=300)
plt.show()